In [30]:
##cell----------01-------------------##
!pip install plyfile
import glob
import numpy as np
from plyfile import PlyData
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, jaccard_score


In [31]:
##cell----------02-------------------##
KITTI360_LABELS = [
    (0,  "unlabeled",       "void"),
    (1,  "ego vehicle",     "void"),
    (2,  "rectification border", "void"),
    (3,  "out of roi",      "void"),
    (4,  "static",          "void"),
    (5,  "dynamic",         "void"),
    (6,  "ground",          "flat"),
    (7,  "road",            "flat"),
    (8,  "sidewalk",        "flat"),
    (9,  "parking",         "flat"),
    (10, "rail track",      "flat"),
    (11, "building",        "construction"),
    (12, "wall",            "construction"),
    (13, "fence",           "construction"),
    (14, "guard rail",      "construction"),
    (15, "bridge",          "construction"),
    (16, "tunnel",          "construction"),
    (17, "pole",            "object"),
    (18, "polegroup",       "object"),
    (19, "traffic light",   "object"),
    (20, "traffic sign",    "object"),
    (21, "vegetation",      "nature"),
    (22, "terrain",         "nature"),
    (23, "sky",             "sky"),
    (24, "person",          "human"),
    (25, "rider",           "human"),
    (26, "car",             "vehicle"),
    (27, "truck",           "vehicle"),
    (28, "bus",             "vehicle"),
    (29, "caravan",         "vehicle"),
    (30, "trailer",         "vehicle"),
    (31, "train",           "vehicle"),
    (32, "motorcycle",      "vehicle"),
    (33, "bicycle",         "vehicle"),
    (34, "garage",          "construction"),
    (35, "gate",            "construction"),
    (36, "stop",            "construction"),
    (37, "smallpole",       "object"),
    (38, "lamp",            "object"),
    (39, "trash bin",       "object"),
    (40, "vending machine", "object"),
    (41, "box",             "object"),
    (42, "unknown construction", "construction"),
    (43, "unknown vehicle", "vehicle"),
    (44, "unknown object",  "object"),
]
IGNORE_IDS = {0, 1, 2, 3, 4, 5}
ROWS = [(i, n, c) for (i, n, c) in KITTI360_LABELS if i not in IGNORE_IDS]
ID_TO_NAME = {i: n for i, n, c in ROWS}
ID_TO_CATEGORY = {i: c for i, n, c in ROWS}
CATEGORY_LIST = sorted(set(c for _, _, c in ROWS))
VALID_IDS = sorted(ID_TO_NAME.keys())
ID_REMAP = {old: new for new, old in enumerate(VALID_IDS)}


In [32]:
##cell----------03-------------------##
def load_ply_points(path, max_points=None, rng=None):
    ply = PlyData.read(path)
    v = ply["vertex"]
    x = np.asarray(v["x"]); y = np.asarray(v["y"]); z = np.asarray(v["z"])
    r = np.asarray(v["red"]); g = np.asarray(v["green"]); b = np.asarray(v["blue"])
    sem = np.asarray(v["semantic"])
    mask = np.isin(sem, VALID_IDS)
    x, y, z, r, g, b, sem = x[mask], y[mask], z[mask], r[mask], g[mask], b[mask], sem[mask]
    if max_points is not None and len(x) > max_points:
        chooser = rng.choice if rng is not None else np.random.choice
        idx = chooser(len(x), max_points, replace=False)
        x, y, z, r, g, b, sem = x[idx], y[idx], z[idx], r[idx], g[idx], b[idx], sem[idx]
    # Center XYZ per window so coordinates are translation-invariant across street blocks
    xyz = np.stack([x, y, z], axis=1).astype(np.float32)
    xyz = xyz - xyz.mean(axis=0, keepdims=True)
    rgb = np.stack([r / 255.0, g / 255.0, b / 255.0], axis=1).astype(np.float32)
    feats = np.concatenate([xyz, rgb], axis=1)
    labels = np.array([ID_REMAP[s] for s in sem], dtype=np.int64)
    return feats, labels


def load_kitti360_train_test(ply_glob, max_points_per_file=20000, test_size=0.2, seed=0):
    files = sorted(glob.glob(ply_glob))
    assert len(files) >= 2, f"Need at least 2 .ply files to split, found {len(files)} for {ply_glob}"
    train_files, test_files = train_test_split(files, test_size=test_size, random_state=seed)
    rng = np.random.default_rng(seed)

    def _load_list(file_list):
        feats_all, labels_all = [], []
        for path in file_list:
            f, l = load_ply_points(path, max_points=max_points_per_file, rng=rng)
            if len(f) > 0:
                feats_all.append(f)
                labels_all.append(l)
        return np.concatenate(feats_all, axis=0), np.concatenate(labels_all, axis=0)

    X_train, y_train = _load_list(train_files)
    X_test, y_test = _load_list(test_files)

    # Fit XYZ scale strictly on train set to prevent normalization leakage
    sigma = X_train[:, :3].std(axis=0, keepdims=True) + 1e-6
    X_train[:, :3] /= sigma
    X_test[:, :3] /= sigma
    return X_train, X_test, y_train, y_test


In [33]:
##cell----------04-------------------##
class Tree:
    def __init__(self, levels, paths):
        self.levels = levels
        self.paths = paths

    @property
    def depth(self):
        return len(self.levels)

    def level_width(self, l):
        return len(self.levels[l])

    def total_width(self):
        return sum(self.level_width(l) for l in range(self.depth))

    def flat_width(self):
        return self.level_width(self.depth - 1)

    def level_index(self, l, name):
        return self.levels[l].index(name)


def build_kitti360_tree():
    leaf_names = [ID_TO_NAME[i] for i in VALID_IDS]
    category_names = CATEGORY_LIST
    levels = [category_names, leaf_names]
    paths = {}
    for i in VALID_IDS:
        name = ID_TO_NAME[i]
        cat = ID_TO_CATEGORY[i]
        paths[name] = [cat, name]
    return Tree(levels=levels, paths=paths)


In [34]:
##cell----------05-------------------##
def make_backbone(in_dim, hidden_dim=128):
    return nn.Sequential(
        nn.Linear(in_dim, hidden_dim),
        nn.LayerNorm(hidden_dim),
        nn.GELU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.LayerNorm(hidden_dim),
        nn.GELU(),
    )


class HierarchicalHead(nn.Module):
    def __init__(self, in_dim, level_widths, hidden_dim=128):
        super().__init__()
        self.backbone = make_backbone(in_dim, hidden_dim)
        self.level_widths = level_widths
        self.level_heads = nn.ModuleList([nn.Linear(hidden_dim, n) for n in level_widths])
        self.F = nn.Linear(sum(level_widths), level_widths[-1])

    def forward(self, x):
        feat = self.backbone(x)
        level_logits = [head(feat) for head in self.level_heads]
        level_probs = [torch.softmax(z, dim=-1) for z in level_logits]
        h = torch.cat(level_probs, dim=-1)
        flat_logits = self.F(h)
        return level_logits, level_probs, flat_logits

    def total_width(self):
        return sum(self.level_widths)


class FlatHead(nn.Module):
    def __init__(self, in_dim, num_classes, hidden_dim=128):
        super().__init__()
        self.backbone = make_backbone(in_dim, hidden_dim)
        self.linear = nn.Linear(hidden_dim, num_classes)
        self.num_classes = num_classes

    def forward(self, x):
        return self.linear(self.backbone(x))

    def total_width(self):
        return self.num_classes


In [35]:
##cell----------06-------------------##
def inter_level_loss(level_logits, level_label_ids):
    loss = 0.0
    for logits, labels in zip(level_logits, level_label_ids):
        loss = loss + F.cross_entropy(logits, labels)
    return loss


def cross_level_loss(flat_logits, flat_labels):
    return F.cross_entropy(flat_logits, flat_labels)


def semantic_loss(level_logits, level_label_ids, flat_logits, flat_labels, epoch, eta=15, omega1=1.0, omega2_after=5.0):
    omega2 = 0.0 if epoch < eta else omega2_after
    l_inter = inter_level_loss(level_logits, level_label_ids)
    l_cross = cross_level_loss(flat_logits, flat_labels)
    total = omega1 * l_inter + omega2 * l_cross
    return total, l_inter.item(), l_cross.item(), omega2


In [36]:
##cell----------07-------------------##
def leaf_labels_to_level_ids(tree, leaf_labels):
    leaf_names = tree.levels[-1]
    level_ids = []
    for l in range(tree.depth):
        lut = np.array(
            [tree.level_index(l, tree.paths[name][l]) for name in leaf_names],
            dtype=np.int64,
        )
        level_ids.append(lut[leaf_labels])
    return level_ids


def train_hierarchical(tree, X, y, in_dim, epochs=60, batch_size=512, lr=1e-3, eta=15, device="cpu"):
    level_ids_all = leaf_labels_to_level_ids(tree, y)
    level_widths = [tree.level_width(l) for l in range(tree.depth)]
    model = HierarchicalHead(in_dim, level_widths).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    X_t = torch.from_numpy(X)
    level_t = [torch.from_numpy(ids) for ids in level_ids_all]
    loader = DataLoader(TensorDataset(X_t, *level_t), batch_size=batch_size, shuffle=True)
    model.train()
    for epoch in range(epochs):
        for batch in loader:
            xb = batch[0].to(device, non_blocking=True)
            level_labels_b = [b.to(device, non_blocking=True) for b in batch[1:]]
            flat_labels_b = level_labels_b[-1]
            level_logits, _, flat_logits = model(xb)
            loss, _, _, _ = semantic_loss(
                level_logits, level_labels_b, flat_logits, flat_labels_b, epoch=epoch, eta=eta
            )
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


def train_flat(num_classes, X, y, in_dim, epochs=60, batch_size=512, lr=1e-3, device="cpu"):
    model = FlatHead(in_dim, num_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


In [37]:
##cell----------08-------------------##
@torch.no_grad()
def evaluate_hierarchical(model, tree, X, y, device="cpu", batch_size=4096):
    model.eval()
    level_preds = [[] for _ in range(tree.depth)]
    flat_preds = []
    for start in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).to(device)
        level_logits, _, flat_logits = model(xb)
        for l, logits in enumerate(level_logits):
            level_preds[l].append(logits.argmax(dim=-1).cpu().numpy())
        flat_preds.append(flat_logits.argmax(dim=-1).cpu().numpy())

    level_ids_all = leaf_labels_to_level_ids(tree, y)
    per_level_acc = [
        accuracy_score(level_ids_all[l], np.concatenate(level_preds[l]))
        for l in range(tree.depth)
    ]
    flat_pred = np.concatenate(flat_preds)
    flat_acc = accuracy_score(y, flat_pred)
    gt_labels = sorted(set(y.tolist()))
    miou = jaccard_score(y, flat_pred, labels=gt_labels, average="macro", zero_division=0)
    return {"per_level_acc": per_level_acc, "leaf_acc": flat_acc, "mIoU": miou}


@torch.no_grad()
def evaluate_flat(model, X, y, device="cpu", batch_size=4096):
    model.eval()
    preds = []
    for start in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).to(device)
        preds.append(model(xb).argmax(dim=-1).cpu().numpy())
    pred = np.concatenate(preds)
    acc = accuracy_score(y, pred)
    gt_labels = sorted(set(y.tolist()))
    miou = jaccard_score(y, pred, labels=gt_labels, average="macro", zero_division=0)
    return {"acc": acc, "mIoU": miou}


In [38]:
##cell----------09-------------------##
# Cell 9
PLY_GLOB = "/kaggle/input/datasets/pzelner6900/kitti-360-prepared-dataset/KITTI360/data_3d_semantics/2013_05_28_drive_0000_sync/static/*.ply"
MAX_POINTS_PER_FILE = 20000
EPOCHS = 60
ETA = 15
BATCH_SIZE = 512
LR = 1e-3
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED)
np.random.seed(SEED)



In [39]:
#cell10
tree = build_kitti360_tree()
print(tree.flat_width(), tree.total_width())


39 46


In [40]:
#cell11
# Cell 11
X_train, X_test, y_train, y_test = load_kitti360_train_test(
    PLY_GLOB, max_points_per_file=MAX_POINTS_PER_FILE, test_size=0.2, seed=SEED
)
in_dim = X_train.shape[1]
num_classes = tree.flat_width()
print(f"Device: {DEVICE} | Train points: {len(X_train)} | Test points: {len(X_test)}")



Device: cuda | Train points: 700000 | Test points: 180000


In [41]:

hmodel = train_hierarchical(tree, X_train, y_train, in_dim, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, eta=ETA, device=DEVICE)
fmodel = train_flat(num_classes, X_train, y_train, in_dim, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)


In [42]:

h_metrics = evaluate_hierarchical(hmodel, tree, X_test, y_test, device=DEVICE)
f_metrics = evaluate_flat(fmodel, X_test, y_test, device=DEVICE)
print(f_metrics)
print(h_metrics)


{'acc': 0.5346333333333333, 'mIoU': np.float64(0.08618847589379312)}
{'per_level_acc': [0.7754777777777778, 0.5430388888888888], 'leaf_acc': 0.5303111111111111, 'mIoU': np.float64(0.08766907848527897)}
